# PydanticAI

**Domain:** Agentic AI  ·  **recommended addition**  ·  **runnable:** yes

A refresher on **PydanticAI** — the agent framework from the team behind [Pydantic](https://docs.pydantic.dev/). Its pitch: bring the *"FastAPI feeling"* to building LLM agents. You define an `Agent`, optionally a **typed output schema** and **typed dependencies**, register **tools** as plain Python functions, and PydanticAI handles the model call, the tool-call loop, output validation, and retries — while keeping everything **statically typed** so your editor and type-checker understand the whole flow.

It sits in the same space as [[langchain]], [[langgraph]], [[crewai]], and [[smolagents]], but its differentiator is **type safety + validation as a first-class concern** rather than an add-on.

## 1. What & Why

Most LLM apps don't fail because the model is dumb — they fail at the *seams*: the model returns prose when you wanted JSON, the JSON is missing a field, a tool gets called with the wrong argument types, and you discover all of this at runtime with a stack trace deep inside a string-parsing helper.

**PydanticAI's bet:** if you describe what you want with Pydantic models and Python type hints, the framework can *enforce* it. The agent's output is validated into a Pydantic model (with automatic re-prompting if validation fails), tool arguments are validated against the function's signature, and your dependencies are injected with a known type. The result is LLM code that behaves like the rest of your typed Python codebase.

**The problem it solves**

- **Structured, validated output** — declare `output_type=SomeModel` and you get back a real, validated instance, not a string you have to `json.loads` and pray over. On a validation error PydanticAI feeds the error back to the model and retries.
- **Typed tools with zero boilerplate** — decorate a function with `@agent.tool`; PydanticAI builds the JSON schema from its signature and docstring, validates the model's arguments, and runs it.
- **Dependency injection** — pass a typed `deps` object (DB connection, API client, config) into every tool and dynamic prompt via `RunContext`, so tools stay pure and testable.
- **Model-agnostic** — the same `Agent` runs against Anthropic, OpenAI, Gemini, Groq, Bedrock, Ollama, and more by changing a model string.
- **Genuinely testable** — `TestModel` and `FunctionModel` let you exercise the *entire* agent loop with **no API key and no network**, which is exactly what makes the examples below run offline.

**When to reach for it:** you're building a production agent in Python and you care about types, validation, and testability — especially if you already use Pydantic/FastAPI. **When not to:** you need a large catalog of pre-built integrations/chains ([[langchain]]), an explicit graph with durable state and human-in-the-loop ([[langgraph]]), or a multi-agent role-play orchestration ([[crewai]]). PydanticAI is deliberately lean and single-agent-first.

## 2. Mental Model

**PydanticAI is FastAPI for LLM calls.**

In FastAPI you write a typed function, and the framework turns request data into validated arguments and your return value into a validated response — you never hand-parse the HTTP body. PydanticAI does the same for a *model call*:

```
        deps (typed)                       output_type (Pydantic)
            │                                       ▲
            ▼                                       │  validated, retried
   ┌─────────────────────────────────────────────────────────┐
   │                        Agent                              │
   │   instructions  +  user prompt  ──►  LLM                  │
   │                         ▲   │                             │
   │            tool result  │   ▼  tool call (typed args)     │
   │                    ┌──────────────┐                       │
   │                    │  @agent.tool │  ◄── RunContext.deps  │
   │                    └──────────────┘                       │
   └─────────────────────────────────────────────────────────┘
```

The `Agent` object is the analogue of a FastAPI `app`: you define it **once** (usually as a module global) with its model, instructions, output type, and tools, then call `.run()` / `.run_sync()` many times. The loop in the middle — *call model → maybe call a tool → feed result back → repeat until a final validated output* — is run for you. Your job is just to declare the shapes (`deps_type`, `output_type`) and the tools.

## 3. Key Concepts

| Concept | What it is |
|---|---|
| **`Agent`** | The central object. Generic over its deps and output types (`Agent[Deps, Output]`). Created once, reused for many runs. Holds the model, instructions, tools, and output schema. |
| **`output_type`** | A Pydantic model (or scalar/`Union`) describing the agent's final answer. PydanticAI forces the model to produce matching data, validates it, and **re-prompts on failure**. Without it you just get text. |
| **`instructions` / `system_prompt`** | The agent's standing guidance. `instructions` is the modern field (not carried across in message history); use `@agent.system_prompt` / a function for **dynamic** prompts that read `deps`. |
| **Tools** | Python functions the model may call, registered via `@agent.tool` (gets `RunContext`) or `@agent.tool_plain` (no context). The schema comes from the **type hints + docstring** — keep both crisp. |
| **`RunContext[Deps]`** | The object passed to tools and dynamic prompts. `ctx.deps` is your injected dependencies; it also exposes usage, retry count, and message history. |
| **`deps_type` / dependency injection** | Declare a type (often a `@dataclass`) of external things tools need — DB handle, HTTP client, config. Pass an instance per run via `deps=...`. Keeps tools pure and unit-testable. |
| **`run()` / `run_sync()` / `run_stream()`** | Execute the agent: async, sync wrapper, and streaming. All return a result whose `.output` is your validated `output_type` and `.usage` reports token counts. |
| **Output / result validators** | `@agent.output_validator` runs *after* schema validation for business rules; raise `ModelRetry` to send the model back for another attempt. |
| **`ModelRetry`** | Raise it from a tool or validator to tell the model *"that was wrong, here's why, try again"* — the framework loops with the error as feedback (bounded by `retries`). |
| **`TestModel` / `FunctionModel`** | Fake models for tests/dev. `TestModel` auto-generates schema-valid output and calls every tool once; `FunctionModel` lets you script the model's behavior exactly — both **offline, no key**. |

## 4. Setup

PydanticAI ships as two distributions:

- **`pydantic-ai`** — the full package with every model provider.
- **`pydantic-ai-slim`** — the lean core; add provider extras you actually need (e.g. `pydantic-ai-slim[anthropic]`).

```bash
# Full install (simplest):
pip install pydantic-ai

# Or slim core + just the provider you use:
pip install "pydantic-ai-slim[anthropic]"

# Optional: first-class observability via the Pydantic team's tracing tool
pip install logfire
```

`TestModel` and `FunctionModel` (used in Examples 1 & 2) live in the **slim core** and need no provider and no API key. Example 3 calls a real Claude model and needs the `anthropic` extra plus `ANTHROPIC_API_KEY`. The cell below just reports what's available — nothing is required for the offline examples.

In [1]:
# Environment check. Examples 1 & 2 are fully offline; Example 3 is gated.
import importlib.util, os

has_pai = importlib.util.find_spec("pydantic_ai") is not None
has_anthropic = importlib.util.find_spec("anthropic") is not None
has_key = bool(os.getenv("ANTHROPIC_API_KEY"))

print("pydantic_ai installed:    ", has_pai)
print("anthropic provider:       ", has_anthropic)
print("ANTHROPIC_API_KEY present:", has_key)
print()
print("Examples 1 & 2 run offline with TestModel / FunctionModel (no key, no network).")
print("Example 3 runs only if the anthropic extra is installed AND the key is set.")

pydantic_ai installed:     True
anthropic provider:        False
ANTHROPIC_API_KEY present: False

Examples 1 & 2 run offline with TestModel / FunctionModel (no key, no network).
Example 3 runs only if the anthropic extra is installed AND the key is set.


## 5. Worked Examples

The first two examples run **completely offline** using PydanticAI's fake models, so the whole agent machinery — schema enforcement, the tool-call loop, dependency injection — executes deterministically with no API key. Example 3 swaps in a real Claude model behind an env-var gate.

### Example 1 — Typed, validated output

The headline feature: declare a Pydantic `output_type` and the agent hands you back a **validated instance**, not a string. We triage a support message into a `Ticket`.

We drive it with `TestModel`, which generates schema-valid output offline. To get a realistic object (instead of placeholder data), we hand it `custom_output_args` — but note the *contract* is identical with a real model: `result.output` is always a fully-validated `Ticket`, and `priority` is guaranteed to be an `int`.

> We use `await agent.run(...)` rather than `agent.run_sync(...)` because a Jupyter kernel already has a running event loop (see Gotchas). In a plain script, `run_sync()` is the convenient choice.

In [2]:
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from pydantic_ai.models.test import TestModel


class Ticket(BaseModel):
    """A triaged support ticket."""
    summary: str = Field(description="One-line summary of the issue")
    sentiment: str
    priority: int = Field(description="1 (low) .. 5 (urgent)", ge=1, le=5)


# Define the agent ONCE. `output_type=Ticket` makes the agent return a Ticket.
triage_agent = Agent(
    output_type=Ticket,
    instructions="Triage the support message into a Ticket.",
)

# Offline: TestModel produces schema-valid data. We supply realistic values so the
# output reads naturally; with a real model you'd just pass model="anthropic:...".
fake = TestModel(custom_output_args={
    "summary": "Cannot log in after password reset",
    "sentiment": "frustrated",
    "priority": 4,
})

result = await triage_agent.run(
    "I reset my password and now I'm completely locked out. Fix this now!",
    model=fake,
)

ticket = result.output
print("type:        ", type(ticket).__name__)
print("ticket:      ", ticket)
print("priority+1:  ", ticket.priority + 1, "(it's a real int, not a string)")
print("usage:       ", result.usage)

type:         Ticket
ticket:       summary='Cannot log in after password reset' sentiment='frustrated' priority=4
priority+1:   5 (it's a real int, not a string)
usage:        RunUsage(input_tokens=63, output_tokens=13, requests=1)


### Example 2 — Tools + dependency injection (the agent loop)

Now the part that makes it an *agent*: tools and the call loop. We build a currency-conversion agent whose exchange rates live in an injected `deps` object (imagine a live rates API or DB). The tool reads them via `RunContext`.

To make the loop deterministic and offline we use `FunctionModel`: we supply a plain function that plays the role of the LLM. It does what a real model would — on the first turn it emits a **tool call** (`convert`), and after seeing the **tool result** it emits the final text answer. PydanticAI runs the actual tool (validating arguments against the type hints) and threads the result back automatically.

In [3]:
from dataclasses import dataclass
from pydantic_ai import Agent, RunContext
from pydantic_ai.models.function import FunctionModel, AgentInfo
from pydantic_ai.messages import (
    ModelMessage, ModelResponse, TextPart, ToolCallPart, ToolReturnPart,
)


@dataclass
class Deps:
    """Things the tools need at runtime — injected per run, never hard-coded."""
    rates: dict[str, float]  # rate vs. USD


fx_agent = Agent(deps_type=Deps, instructions="Convert currency using the tool.")


@fx_agent.tool
def convert(ctx: RunContext[Deps], amount: float, to: str) -> str:
    """Convert `amount` USD into currency `to` using injected rates."""
    rate = ctx.deps.rates[to]
    return f"{amount * rate:.2f} {to}"


# A scripted stand-in for the LLM: call the tool, then answer from its result.
def model_logic(messages: list[ModelMessage], info: AgentInfo) -> ModelResponse:
    last = messages[-1]
    tool_return = next(
        (p.content for p in last.parts if isinstance(p, ToolReturnPart)), None
    )
    if tool_return is not None:                       # 2nd turn: we have the result
        return ModelResponse(parts=[TextPart(f"That's {tool_return}.")])
    return ModelResponse(parts=[                      # 1st turn: ask for a tool call
        ToolCallPart("convert", {"amount": 100.0, "to": "EUR"})
    ])


result = await fx_agent.run(
    "Convert 100 USD to EUR.",
    model=FunctionModel(model_logic),
    deps=Deps(rates={"EUR": 0.92, "GBP": 0.79}),
)

print("OUTPUT:", result.output)
print("\n--- full message trace (this is the agent loop) ---")
for m in result.all_messages():
    for p in m.parts:
        kind = type(p).__name__
        detail = getattr(p, "content", None) or getattr(p, "args", "")
        print(f"  {kind:16} {detail}")

OUTPUT: That's 92.00 EUR.

--- full message trace (this is the agent loop) ---
  UserPromptPart   Convert 100 USD to EUR.
  ToolCallPart     {'amount': 100.0, 'to': 'EUR'}
  ToolReturnPart   92.00 EUR
  TextPart         That's 92.00 EUR.


### Example 3 — The same agent against a real model (gated)

Nothing about the agent definition changes to go live — you only swap the *model*. Here we point `triage_agent` from Example 1 at a real Claude model via the `"anthropic:..."` model string. PydanticAI handles forcing structured output and validating the response into a `Ticket`.

This cell runs only if the `anthropic` extra is installed **and** `ANTHROPIC_API_KEY` is set; otherwise it prints the call shape and skips, so the notebook still executes top-to-bottom.

In [4]:
if has_anthropic and has_key:
    # Reuse the SAME agent from Example 1 — only the model string is new.
    live = await triage_agent.run(
        "I reset my password and now I'm completely locked out. Fix this now!",
        model="anthropic:claude-haiku-4-5",   # small + cheap is plenty for triage
    )
    print("Validated Ticket from a real model:")
    print("  ", live.output)
    print("  usage:", live.usage)
else:
    print("Skipping live call (need the anthropic extra + ANTHROPIC_API_KEY).")
    print("With both set, this would run:")
    print('    await triage_agent.run(msg, model="anthropic:claude-haiku-4-5")')
    print("and return a fully-validated Ticket — identical contract to Example 1,")
    print("but the field values would come from Claude instead of TestModel.")

Skipping live call (need the anthropic extra + ANTHROPIC_API_KEY).
With both set, this would run:
    await triage_agent.run(msg, model="anthropic:claude-haiku-4-5")
and return a fully-validated Ticket — identical contract to Example 1,
but the field values would come from Claude instead of TestModel.


## 6. Gotchas & Pitfalls

- **Recreating the `Agent` on every request.** The `Agent` is meant to be a long-lived module global, like a FastAPI `app`. Building it inside your request handler wastes work and muddies the design. Define once, `run()` many times with different `deps`.
- **`async` vs `sync`.** `run()` is a coroutine; in a script or notebook you'll often want `run_sync()`. But `run_sync()` **cannot be called inside a running event loop** (e.g. another `async` function, or some notebook setups) — use `await agent.run(...)` there instead.
- **Thin tool docstrings / type hints.** The tool's JSON schema is generated *entirely* from its signature and docstring. Vague names, missing types, or no docstring → the model calls the tool wrongly. Treat the signature + docstring as the prompt they are.
- **Forgetting validation can retry — and that retries cost calls.** A failed `output_type` validation or a `ModelRetry` raised in a tool sends the model back for another attempt. That's powerful, but each retry is another model call; cap it with the `retries=` argument and don't let a flaky validator loop.
- **API churn / version pinning.** PydanticAI iterated fast pre-1.0: `result_type` was renamed to `output_type`, `@agent.result_validator` to `@agent.output_validator`, and `RunResult.data` to `.output`. Pin a version and check the docs for the exact names in *your* release (this notebook targets the 2.x line).
- **`TestModel` calls every tool by default.** It's designed to exercise your whole agent, so it will invoke each registered tool once with generated arguments. Great for smoke tests, surprising if you expected it to call nothing — use `FunctionModel` when you need precise control over what the "model" does.
- **`instructions` vs `system_prompt` semantics.** `instructions` are *not* preserved when you pass prior `message_history` into a new run, whereas `system_prompt` content is. If multi-turn behavior surprises you, check which one you used.

## 7. When to Use vs Alternatives

| Option | Best for | Trade-offs vs PydanticAI |
|---|---|---|
| **PydanticAI (this)** | Typed, validated, testable single-agent apps in Python; teams already on Pydantic/FastAPI | Lean and single-agent-first; fewer prebuilt integrations than the big frameworks |
| **[[langchain]]** | Huge ecosystem of prebuilt chains, loaders, vector-store and tool integrations | Heavier, looser typing, more abstraction layers; PydanticAI is smaller and more type-safe |
| **[[langgraph]]** | Explicit stateful graphs: branching, cycles, durable persistence, human-in-the-loop | More machinery to wire up; PydanticAI's loop is implicit and simpler but less controllable |
| **[[crewai]] / [[autogen]]** | Multi-agent role/conversation orchestration out of the box | PydanticAI focuses on one solid agent; you compose multi-agent flows yourself |
| **[[smolagents]]** | Minimal code-writing agents (model emits Python to run) | Different paradigm (code-as-action); PydanticAI is tool-calling + structured output |
| **Raw provider SDK + Pydantic** | Maximum control, minimal deps, simple one-shot structured calls | You hand-roll the tool loop, retries, validation feedback; PydanticAI gives you that for free |
| **Instructor** | Just want structured/validated *output* from one call, no agent loop | Instructor is output-only; PydanticAI adds tools, deps, and the multi-step agent loop |

**Rule of thumb:** reach for PydanticAI when you want an agent that behaves like the rest of your typed Python — validated I/O, injected dependencies, real unit tests — without buying into a large framework. If you need a sprawling integration catalog, an explicit state graph, or multi-agent choreography, the frameworks to its left fit better.

## 8. Resources

- **Official docs — PydanticAI** — https://ai.pydantic.dev/
- **Agents guide (output types, instructions, the run loop)** — https://ai.pydantic.dev/agents/
- **Tools & dependency injection** — https://ai.pydantic.dev/tools/  ·  https://ai.pydantic.dev/dependencies/
- **Testing & evals (TestModel / FunctionModel)** — https://ai.pydantic.dev/testing/
- **GitHub repository (source + examples)** — https://github.com/pydantic/pydantic-ai
- **Pydantic Logfire (observability for agents)** — https://logfire.pydantic.dev/docs/